Task: For small $n$ and $q$, implement an exhaustive secret search, i.e., for every candidate $s'$, calculate $\vec{r}=\vec{b}-{\bf A}\vec{s}'$ and convert the residues to their centered representatives. Does an attacker gain advantage from knowing the true error distribution?

**This file will be rather heavily created with Copilot.** Sorry): All the following cells are Copilot-made.

In [38]:
# Creating a toy LWE oracle

from sage.crypto.lwe import LWE
from sage.probability.probability_distribution import RealDistribution
# Toy parameters
n = 4
q = 11
m = 20
# Rounded Gaussian sampler
class CustomSampler:
    def __init__(self, D):
        self.D = D
    def __call__(self):
        return round(self.D.get_random_element())
D = RealDistribution('gaussian', 1.0)
lwe = LWE(
    n=n,
    q=q,
    D=CustomSampler(D),
    secret_dist='uniform',
    m=None
)

In [39]:
# Get true s for comparison
s_true = lwe._LWE__s
print("True secret:", s_true)

True secret: (5, 6, 1, 5)


In [40]:
# Generate samples
S = [lwe() for _ in range(m)]
# Split into matrix-vector forms
Rq = Zmod(q)
A = matrix(Rq, [sample[0] for sample in S])
b = vector(Rq, [sample[1] for sample in S])
print("A dimensions:", A.dimensions())
print("b length:", len(b))

A dimensions: (20, 4)
b length: 20


In [41]:
# Centering
def centered(x, q):
    x = int(x) % q
    if x > q//2:
        x -= q
    return x

In [42]:
# Scoring based on residual: For Gaussian errors the ML score is equivalent to minimizing \sum_i r_i^2
def residual_norm_squared(r):
    return sum(x*x for x in r)

In [43]:
r = [4, 2, 7]

In [44]:
residual_norm_squared(r)

69

In [45]:
from itertools import product
len(list(product(range(q), repeat=n)))

14641

In [46]:
# Exhaustive search
best_s = None
best_score = Infinity
print(f'have to look through {len(list(product(range(q), repeat=n)))} combinations')
for coords in product(range(q), repeat=n):
    s_candidate = vector(Rq, coords)
    r = b - A*s_candidate
    r_centered = [centered(x, q) for x in r]
    score = residual_norm_squared(r_centered)
    if score < best_score:
        best_score = score
        best_s = s_candidate

print("Recovered secret:", best_s)
print("Best score:", best_score)

have to look through 14641 combinations
Recovered secret: (5, 6, 1, 5)
Best score: 23


In [47]:
# Compare with truth
print("True secret     :", s_true)
print("Recovered secret:", best_s)
print("Success:", s_true == best_s)

True secret     : (5, 6, 1, 5)
Recovered secret: (5, 6, 1, 5)
Success: True


In [48]:
# Inspect top candidates
scores = []
for coords in product(range(q), repeat=n):
    s_candidate = vector(Rq, coords)
    r = b - A*s_candidate
    r_centered = [centered(x, q) for x in r]
    score = residual_norm_squared(r_centered)
    scores.append((score, s_candidate))

scores.sort()
for score, secret in scores[:10]:
    print(score, secret)

23 (5, 6, 1, 5)
61 (5, 1, 9, 10)
72 (9, 5, 6, 8)
75 (0, 8, 4, 8)
77 (0, 6, 8, 3)
78 (0, 6, 3, 1)
82 (9, 0, 3, 2)
83 (8, 4, 10, 4)
83 (10, 9, 9, 1)
84 (9, 4, 9, 8)


In [49]:
# Testing different error distributions: replace D with following lines:
D = RealDistribution('uniform', [-3,3])
D = RealDistribution('t', 2)
D = RealDistribution('exppow', [1,4])